<a href="https://colab.research.google.com/github/aWolander/google-colab/blob/main/Federated%20Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from __future__ import annotations
import copy
import torch
import time
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets
from torchvision.transforms import ToTensor, Normalize, Compose, Resize, CenterCrop
import matplotlib.pyplot as plt
import random
import numpy as np
import math
from tqdm.notebook import trange, tqdm
try:
    device = torch.accelerator.current_accelerator().type if \
    torch.accelerator.is_available() else "cpu" # i get an error when i run this locally
except:
    device="cpu"

transform = Compose([
    #Resize(224), # slows down process significantly
    #CenterCrop(224),
    ToTensor(),
    Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
]) # https://github.com/facebookresearch/dino/blob/7c446df5b9f45747937fb0d72314eb9f7b66930a/eval_image_retrieval.py#L106

CIFAR100 = datasets.CIFAR100(
  root="data",
  download=True,
  transform=transform
)


100%|██████████| 169M/169M [00:13<00:00, 12.6MB/s]


In [4]:
def main() -> None:
    """
    Loads CIFAR-100 dataset and splits it into training, validation, and test sets.
    Then performs an IID client split and tests the data distribution.
    """
    K = 5  # number of clients
    N = 10
    CIFAR100 = datasets.CIFAR100(
        root="data",
        download=True,
        transform=ToTensor()
    )

    train_dataset, validate_dataset, test_dataset = torch.utils.data.random_split(
        CIFAR100, [0.7, 0.15, 0.15]
    )

    client_datasets = split_data_non_iid(train_dataset, K, N)
    test_split(client_datasets)


def create_label_indexing(dataset: Dataset) -> dict[int, list[int]]:
    """
    Creates a dictionary mapping each class label to the list of indices where it appears.

    Args:
        dataset: Dataset object (e.g. CIFAR100 or Subset).

    Returns:
        A dictionary mapping label -> list of indices.
    """
    label_index = {i: [] for i in range(100)}
    for idx, (_, label) in enumerate(dataset):
        label_index[label].append(idx)
    return label_index


def split_data_non_iid(dataset: Dataset, K: int, N_c: int = 100) -> list[Subset]:
    """
    Splits the dataset into K non-IID subsets based on label distribution.

    Args:
        dataset: PyTorch dataset.
        K: Number of clients.
        N_c: Number of labels per client.

    Returns:
        List of K Subsets representing client datasets.
    """
    index_split = [[] for _ in range(K)]
    label_index = create_label_indexing(dataset)
    unused_labels = list(range(100))
    available_labels = {}

    for client in range(K):
        client_labels = random.sample(unused_labels, N_c)
        available_labels[client] = client_labels

    labels_exhausted = [False for _ in range(K)]

    while not all(labels_exhausted):
        for client in range(K):
            if not available_labels[client]:
                labels_exhausted[client] = True
                continue
            for label in available_labels[client][:]:
                if not label_index[label]:
                    available_labels[client].remove(label)
                    continue
                index_split[client].append(label_index[label].pop())

    for indices in index_split:
        random.shuffle(indices)

    return [Subset(dataset, indices) for indices in index_split]

def split(l: list[int], n: int) -> list[list[int]]:
    """
    Splits a list into n parts in a round-robin fashion.

    Args:
        l: List to split.
        n: Number of splits.

    Returns:
        A list of n sublists.
    """
    return [l[i::n] for i in range(n)]


def split_data_iid(dataset: Dataset, K: int) -> list[Subset]:
    """
    Splits the dataset IID among K clients.

    Args:
        dataset: PyTorch dataset.
        K: Number of clients.

    Returns:
        List of K Subsets representing IID client datasets.
    """
    index_split = [[] for _ in range(K)]
    label_index = create_label_indexing(dataset)

    for label in range(100):
        split_indices = split(label_index[label], K)
        for client in range(K):
            index_split[client] += split_indices[client]

    for indices in index_split:
        random.shuffle(indices)

    return [Subset(dataset, indices) for indices in index_split]


def test_split(client_datasets: list[Subset]) -> None:
    """
    Plots the label distribution across client datasets.

    Args:
        client_datasets: List of Subsets, each representing a client.

    Returns:
        None
    """
    bottom = np.zeros(100)

    for client_id, dataset in enumerate(client_datasets):
        occurrences = np.zeros(100)
        for _, label in dataset:
            occurrences[label] += 1

        non_zero = occurrences[occurrences > 0]
        print(
            f"Client {client_id}: "
            f"Classes = {np.count_nonzero(occurrences)}, "
            f"Mean = {non_zero.mean():.2f}, Std = {non_zero.std():.2f}"
        )

        plt.bar(range(100), occurrences, bottom=bottom, label=f"Client {client_id}")
        bottom += occurrences

    plt.xlabel("Class label")
    plt.ylabel("Number of samples")
    plt.title("Client Data Distribution")
    plt.show()

#if "__main__" == __name__:
#    main()



In [5]:
'''This is done seperately so that saving and loading the models does not grant
a different dataset. A bit annoying to have to change K here but what can you do'''
K= 100
N_c = 50

train_dataset, validate_dataset, test_dataset = torch.utils.data.random_split(CIFAR100, [0.7,0.15,0.15])

split_dataset_iid = split_data_iid(train_dataset, K)
split_dataset_non_iid = split_data_non_iid(train_dataset, K, N_c)
split_train_dataloader_iid = [DataLoader(subset, batch_size=100) for subset in split_dataset_iid]
split_train_dataloader_non_iid = [DataLoader(subset, batch_size=100) for subset in split_dataset_non_iid]


train_dataloader = DataLoader(train_dataset, batch_size=100)
test_dataloader = DataLoader(test_dataset, batch_size=100)
validate_dataloader = DataLoader(validate_dataset, batch_size=100)

In [6]:
#import logging

#logging.basicConfig(level=logging.INFO)
#logger = logging.getLogger(__name__)

class DinoFullModel(nn.Module):
    """
    A wrapper around the DINO ViT-S/16 model.
    Adds a custom classification head and includes training and evaluation utilities.
    """

    def __init__(self, learning_rate: float = 1e-3, momentum=0.9, device="cpu", filepath: str|None = None) -> None:
        """
        Adds a custom classification head and includes training and evaluation utilities.
        Initializes the model with a DINO backbone and custom classification head.

        Args:
            learning_rate (float): Learning rate for optimizer.
            epochs (int): Number of training epochs.
        """
        super().__init__()

        self.learning_rate = learning_rate
        self.momentum = momentum
        self.device = device
        print(device)



        # Load pretrained DINO model
        self.backbone = torch.hub.load('facebookresearch/dino:main', 'dino_vits16').to(self.device)
        # Replace head with a new classifier for CIFAR
        self.head = nn.Linear(self.backbone.embed_dim, 100)

        # self.backbone.head = nn.Sequential( # apparantly not necessary
        #     nn.Linear(self.backbone.embed_dim, 256),
        #     nn.ReLU(inplace=True),
        #     nn.Dropout(0.5),
        #     nn.Linear(256, 128),
        #     nn.ReLU(inplace=True),
        #     nn.Dropout(0.5),
        #     nn.Linear(128, 100)
        # ).to(device)

        # Loss and optimizer
        self.loss_fn = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.SGD(self.parameters(), lr=self.learning_rate, momentum = self.momentum)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=30)

        # For gradient masking
        self.scores = []
        self.grad_masks = []
        self.rounds = 1
        self.handles = []

        # Performance tracking
        self.training_loss_history = []
        self.training_accuracy_history = []
        self.validate_loss_history = []
        self.validate_accuracy_history = []

        # For saving and Loading
        self.loss = None
        if filepath:
            self.load_model(filepath)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))

    def train_model(self, dataloader: DataLoader, epochs, validate_dataloader: DataLoader|None = None, filepath: str|False=False) -> None:
        """
        Trains the model with optional validation and checkpointing.
        """
        self.train()
        running_loss, total_correct, total_samples = 0.0, 0, 0
        dataset_size = len(dataloader.dataset)

        # Select a few parameters to track for debugging
        tracked_params = []
        for name, param in self.named_parameters():
             if param.requires_grad and param.numel() > 0: # Only track trainable params
                 tracked_params.append((name, param))
             if len(tracked_params) >= 3: # Track up to 3 parameters
                 break


        for epoch in tqdm(range(1, epochs+1), desc="Epochs"):
            running_loss, total_correct, total_samples = 0.0, 0, 0

            if validate_dataloader:
                val_loss, val_acc = self.test_model(validate_dataloader)
                self.validate_loss_history.append(val_loss)
                self.validate_accuracy_history.append(val_acc)
                print(f"Validation Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}")

            for X, y in tqdm(dataloader, desc="Batches", leave=None):
                X, y = X.to(self.device), y.to(self.device)

                outputs = self(X)
                self.loss = self.loss_fn(outputs, y)

                self.loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()

                running_loss += self.loss.item() * X.size(0)
                total_correct += (outputs.argmax(dim=1) == y).sum().item()
                total_samples += X.size(0)

            epoch_loss = running_loss / total_samples
            epoch_acc = total_correct / total_samples
            self.training_loss_history.append(epoch_loss)
            self.training_accuracy_history.append(epoch_acc)

            print(f"Train Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")


            self.scheduler.step()
            self.current_epoch = epoch
            if filepath and epoch%10==0:
                self.save_model(filepath)

    def TaLoS(self, \
                dataloader: DataLoader, \
                rounds:int, \
                sparsity: float, \
                filepath: str|False=False, \
                q:float = 0) -> None:
        '''
        https://arxiv.org/pdf/2504.02620
        '''
        self.rounds = rounds
        self.freeze_head()
        self.unfreeze_backbone()
        self.eval()
        if sparsity < 1.0:
            for round in tqdm(range(1,self.rounds+1), desc="Finetuning rounds"):
                sparseness = sparsity ** (round/self.rounds)
                self.score_parameters(dataloader)
                if q >= 1:
                    self.test_mask_quantization(sparseness,q)

                if filepath:
                    self.save_model(filepath)
                print("Scoring complete, creating mask.")

                self.create_mask(sparseness)
                self.drop_parameters()
        self.backbone = torch.hub.load('facebookresearch/dino:main', 'dino_vits16').to(self.device)

    def score_parameters(self, dataloader: DataLoader) -> None:
        self.scores = []
        for parameter in self.backbone.parameters():
            self.scores.append(torch.zeros_like(parameter))
        total_samples = 0
        dataset_size = len(dataloader.dataset)

        time_spent = 0.0

        for X, y in tqdm(dataloader,desc="Finetuning batches",leave=None):
            X, y = X.to(self.device), y.to(self.device)
            total_samples += X.size(0)
            logits = self(X)
            outdx = torch.distributions.Categorical(logits=logits).sample().unsqueeze(1).detach()
            samples = logits.gather(1, outdx)

            for datapoint_index in tqdm(range(X.size(0)),desc="Progress in batch", leave=None):
                self.zero_grad()
                torch.autograd.backward(samples[datapoint_index], retain_graph=True) # ???
                for parameter, parameter_score in zip(self.backbone.parameters(), self.scores):
                    if parameter.grad is not None:
                        print(torch.clone(parameter.grad.data.pow(2).detach()).mean())
                        parameter_score += torch.clone(parameter.grad.data.pow(2).detach())

    def create_mask(self, sparseness):
        flattened_score_vector = torch.cat([score_tensor.flatten() for score_tensor in self.scores], dim=0)
        total_parameters = len(flattened_score_vector)
        print(len(flattened_score_vector))

        p = math.floor(sparseness*total_parameters)

        threshold_value = flattened_score_vector.sort(descending=True).values[p]

        self.grad_masks = []
        for score in self.scores:
            mask = (score < threshold_value).float()  # 1 for keep, 0 for freeze
            frozen_parameters = torch.count_nonzero(mask)
            if frozen_parameters >= p:
                break
            #print(mask)
            self.grad_masks.append(mask)

    def apply_gradient_mask(self):
        self.remove_previous_mask()
        for param, mask in zip(self.backbone.parameters(), self.grad_masks):
            if mask.shape != param.shape:
                raise ValueError("Mismatch between mask and parameter shape.")
            if mask is not None:
                # Ensure mask is on the same device as the parameter
                mask = mask.to(param.device)
                handle = param.register_hook(self.gradient_mask_hook(mask))
                self.handles.append(handle)

    def drop_parameters(self):
        for param, mask in zip(self.backbone.parameters(), self.grad_masks):
            if mask.shape != param.shape:
                raise ValueError("Mismatch between mask and parameter shape.")
            if mask is not None:
                mask = mask.to(param.device)
                with torch.no_grad():
                    param.data.mul_(mask)

    def remove_previous_mask(self):
        for handle in self.handles:
            handle.remove()
        self.handles = []

    def gradient_mask_hook(self, m):
        return lambda grad: (grad * m)


    def test_model(self, dataloader: DataLoader) -> tuple[float, float]:
        """
        Evaluates the model on a test/validation dataloader.
        """
        self.eval()
        total_loss, total_correct, total_samples = 0.0, 0, 0

        with torch.no_grad():
            for X, y in dataloader:
                X, y = X.to(device), y.to(device)
                outputs = self(X)
                loss = self.loss_fn(outputs, y)

                total_loss += loss.item() * X.size(0)
                total_correct += (outputs.argmax(dim=1) == y).sum().item()
                total_samples += X.size(0)

        return total_loss / total_samples, total_correct / total_samples

    def get_quantized_scores(self, q):
        # Flatten all scores into one vector
        flattened_scores = torch.cat([s.flatten() for s in self.scores])
        abs_scores = flattened_scores.abs()
        x_max = abs_scores.max()
        x_min = abs_scores.min()

        # Vectorized quantization
        quantized_scores = []
        for score_matrix in self.scores:
            abs_vals = score_matrix.abs()
            x_max = abs_vals.max()
            x_min = abs_vals.min()
            signs = score_matrix.sign()

            normalized = (abs_vals - x_min) / (x_max - x_min + 1e-6)
            normalized_clipped = torch.clamp(normalized, 0, 0.999999)

            # Compute l and probabilities
            l = (normalized_clipped * q).floor()
            prob = normalized_clipped * q - l

            # Stochastic rounding
            rand_vals = torch.rand_like(prob)
            round_up = (rand_vals < prob).float()
            phi_vals = (l + round_up) / q

            # Final quantized values
            quantized = signs * (x_min + (x_max - x_min) * phi_vals)

            quantized_scores.append(quantized)

        return quantized_scores

    def test_mask_quantization(self, sparseness:float, q:float) -> None:
        quantized_scores = self.get_quantized_scores(q)
        #for qscore, score in zip(quantized_scores, self.scores):
            #print(qscore)
            #print(score)
        quantized_scores_flat = torch.cat([score_tensor.flatten() for score_tensor in quantized_scores], dim=0)
        scores_flat = torch.cat([score_tensor.flatten() for score_tensor in self.scores], dim=0)

        total_parameters = len(scores_flat)
        p = math.floor(sparseness*total_parameters)

        threshold_value = scores_flat.sort(descending=True).values[p-1]
        threshold_value_quantized = quantized_scores_flat.sort(descending=True).values[p-1]

        total_different_parameters = 0
        for qscore, score in zip(quantized_scores, self.scores):
            different_parameters = torch.count_nonzero(abs((score < threshold_value).float() - (qscore < threshold_value_quantized).float())).item()
            total_different_parameters += different_parameters
            #print(different_parameters)


        print(f"different_parameters: {total_different_parameters}")
        print(f"Percent wrong: {total_different_parameters/p}")
        print(f"threshold values: {threshold_value} and {threshold_value_quantized}")

    def plot_performance(self) -> None:
        """
        Plots training and validation loss and accuracy curves.
        """
        fig, axs = plt.subplots(1, 2, figsize=(12, 5))
        if self.training_loss_history:
            axs[0].plot(self.training_loss_history, label='Train Loss')
        if self.validate_loss_history:
            axs[0].plot(self.validate_loss_history, label='Val Loss')
        axs[0].set_title('Loss')
        axs[0].legend()
        axs[0].grid(True)
        if self.training_accuracy_history:
            axs[1].plot(self.training_accuracy_history, label='Train Accuracy')
        if self.validate_accuracy_history:
            axs[1].plot(self.validate_accuracy_history, label='Val Accuracy')
        axs[1].set_title('Accuracy')
        axs[1].legend()
        axs[1].grid(True)

        hyperparam_text = (
        f"Learning rate: {self.learning_rate}\n"
        f"Momentum: {self.optimizer.param_groups[0].get('momentum', 'N/A')}\n"
        f"Scheduler: CosineAnnealingLR\n"
        f"Epochs: {len(self.training_loss_history)}"
        )
        # Put it in the top left of the first subplot
        axs[0].text(
            0.02, 0.98, hyperparam_text,
            transform=axs[0].transAxes,
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='lightgray', alpha=0.5)
        )

        plt.tight_layout()
        plt.show()


    def save_model(self, filepath: str) -> None:
        """
        Saves model weights and optimizer state.

        Args:
            filepath (str): Path to save the checkpoint.
        """
        torch.save({
            'model_state_dict': self.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'training_loss_history': self.training_loss_history,
            'training_accuracy_history': self.training_accuracy_history,
            'validate_loss_history': self.validate_loss_history,
            'validate_accuracy_history': self.validate_accuracy_history,
            'scores': self.scores,

        }, filepath)
        print(f"Model saved to {filepath}")

    def recover_model(self, filepath: str) -> None:
        """
        Loads model weights and optimizer state from a checkpoint.

        Args:
            filepath (str): Path to the saved checkpoint.
        """
        checkpoint = torch.load(filepath, map_location=device)
        self.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        # Move optimizer state to the correct device
        for state in self.optimizer.state.values():
            for k, v in state.items():
                if isinstance(v, torch.Tensor):
                    state[k] = v.to(device)

        self.training_loss_history = checkpoint['training_loss_history']
        self.training_accuracy_history = checkpoint['training_accuracy_history']
        self.validate_loss_history = checkpoint['validate_loss_history']
        self.validate_accuracy_history = checkpoint['validate_accuracy_history']
        self.scores = checkpoint['scores']
        print(f"Model loaded from {filepath}")

    def load_model(self, filepath: str) -> None:
        """
        Loads model weights from a checkpoint.

        Args:
            filepath (str): Path to the saved checkpoint.
        """
        checkpoint = torch.load(filepath, map_location=device)
        self.load_state_dict(checkpoint['model_state_dict'])
        self.scores = checkpoint['scores']

    def refresh_optimizer(self):
        self.optimizer = torch.optim.SGD(self.parameters(), lr=self.learning_rate, momentum=self.momentum)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=30) # Also re-initialize scheduler


    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True

    def freeze_head(self):
        for param in self.head.parameters():
            param.requires_grad = False

    def unfreeze_head(self):
        for param in self.head.parameters():
            param.requires_grad = True

In [9]:
class FL_client(DinoFullModel):
    '''
    Basically just the single model, but with addition and scaling functionality.
    To facilitate FedAvg more easily.
    '''
    def __init__(self) -> None:
        # Initialize on CPU
        super().__init__(learning_rate=1e-3, momentum = 0.9)


    def __add__(self, other: FL_client) -> FL_client:
        assert isinstance(other, FL_client)
        # Ensure both clients are on the same device for addition
        assert next(self.parameters()).device == next(other.parameters()).device, "Clients must be on the same device for addition"

        temp_client = FL_client().to(next(self.parameters()).device) # Create temp client on the same device
        temp_client.load_state_dict(self.state_dict())
        with torch.no_grad():
            for (name, param), (_, other_param) in zip(temp_client.named_parameters(), other.named_parameters()):
                param.copy_(param + other_param)
        return temp_client

    def __mul__(self, multiplier: float|int) -> FL_client:
        assert isinstance(multiplier, float|int)
        temp_client = FL_client().to(next(self.parameters()).device) # Create temp client on the same device
        temp_client.load_state_dict(self.state_dict())
        with torch.no_grad():
                for name, param in temp_client.named_parameters():
                    param.copy_(param * multiplier)
        return temp_client

    def __rmul__(self, multiplier: float|int) -> FL_client:
        return self.__mul__(multiplier)

    def __sub__(self, other: FL_client) -> FL_client:
        assert isinstance(other, FL_client)
        return self + (-other)

    def __truediv__(self, divisor: float|int) -> FL_client:
        return self.__mul__(1/divisor)

    def get_model(self, other: FL_client) -> None:
        assert isinstance(other, FL_client)
        with torch.no_grad():
            for (name, param), (_, other_param) in zip(self.named_parameters(), other.named_parameters()):
                param.copy_(other_param)


class FL_server():
    def __init__(self, K: int, filepath:str|None = None) -> None:
        # Server model can stay on the device if it fits
        self.model = DinoFullModel().to(device)

        self.K = K

        self.loss_fn = nn.CrossEntropyLoss()
        self.clients = []
        for i in range(self.K):
            # Initialize clients on CPU to save GPU memory
            temp_client = FL_client() # Initialized on CPU
            self.clients.append(temp_client)

        if filepath:
            self.load_model(filepath)

        self.current_training_round = 0


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

    def FedAvg(self, \
               dataloaders_list: list[DataLoader], \
               FedAvg_rounds: int, \
               client_epochs: int, \
               C: float, \
               validate_dataloader: DataLoader|None = None, \
               filepath: str|None = None) -> None:

        for FedAvg_round in tqdm(range(self.current_training_round, FedAvg_rounds+self.current_training_round), desc="FedAvg round"):
            m = max(int(C*self.K), 1) # Ensure at least one client is selected
            rand_set_clients = random.sample(range(self.K), m)
            client_model_sum = None
            m_t = 0 # Total number of samples from selected clients

            for client_id in tqdm(rand_set_clients, desc="Client"):
                temp_client = self.clients[client_id].to(device)
                temp_loader = dataloaders_list[client_id]

                temp_client.train_model(temp_loader, client_epochs)
                n_k = len(temp_loader.dataset)
                m_t += n_k
                # Easier than instantiating an empty model
                if client_model_sum is None:
                    client_model_sum = n_k*temp_client
                else:
                    client_model_sum += n_k*temp_client
                temp_client.to("cpu") # Move client model back to CPU
                torch.cuda.empty_cache()

            client_model_sum = (1/m_t) * client_model_sum
            self.model.backbone = client_model_sum.backbone.to(device)
            self.model.backbone.head = client_model_sum.backbone.head.to(device)
            for client in self.clients:
                client.get_model(client_model_sum)
            if validate_dataloader:
                val_loss, val_acc = self.test_model(validate_dataloader)
                self.model.validate_loss_history.append(val_loss)
                self.model.validate_accuracy_history.append(val_acc)
                print(f"Validation Loss For Server: {val_loss:.4f}, Accuracy: {val_acc:.4f}")

            if filepath:
                self.save_model(filepath)


    def distributed_TaLoS(self,
                          dataloaders_list: list[DataLoader], \
                          rounds:int, \
                          sparsity:float, \
                          filepath:str|None = None) -> None:
        '''
        https://arxiv.org/pdf/2504.02620
        '''
        for client_id, client in enumerate(self.clients):
            client.to(device) # Move client to device for scoring
            client.TaLoS(dataloaders_list[client_id], rounds, sparsity)
            client.cpu() # Move client back to CPU
            self.save_model(filepath)


    def shared_mask_TaLoS(self, dataloaders_list: list[DataLoader], \
                sparsity:float, \
                rounds:int,\
                q=0,\
                filepath: str|False=False \
                ) -> None:
        '''
        https://arxiv.org/pdf/2504.02620
        '''
        self.rounds = rounds
        self.model.freeze_head()
        self.model.unfreeze_backbone()
        # self.train() # Server model isn't trained directly here

        for round in tqdm(range(rounds), desc="Finetuning rounds"):
            sparseness = sparsity ** (round/rounds)
            client_score_sum = None
            m_t = 0

            for client, client_id in tqdm(enumerate(self.clients), desc="Finetuning for clients"):
                client.to(device) # Move client to device for scoring
                client.score_parameters(dataloaders_list[client_id])
                print("Scoring complete, creating mask.")


                loader = dataloaders_list[client_id]
                temp_scores = client.scores
                if q >= 0:
                    temp_scores = client.get_quantized_scores(q)
                    client.test_mask_quantization(sparseness, q) # we quantize twice for this. computationally wasteful but whatever
                n_k = len(loader.dataset)
                m_t += n_k

                scaled_scores = [n_k*score_matrix.cpu() for score_matrix in temp_scores] # Move to CPU before scaling

                if client_score_sum is None:
                    client_score_sum = scaled_scores
                else:
                    # Assuming scores are lists of tensors, add them element-wise
                    client_score_sum = [s1 + s2 for s1, s2 in zip(client_score_sum, scaled_scores)]

                client.to("cpu") # Move client back to CPU

            average_client_scores = [score_matrix/m_t for score_matrix in client_score_sum]

            for client in self.clients:
                client.scores = average_client_scores # pointers shared here. Should not be a problem
                client.create_mask(sparseness)
                # client.apply_mask() # Mask is applied during training/inference

            if filepath:
                self.save_model(filepath)


    def test_model(self, dataloader: DataLoader) -> tuple[float, float]:
        #print(f"Validation Loss: {avg_loss:.4f}, Correct: {total_correct} out of {len(dataloader)}, Accuracy: {accuracy:.2%}")
        return self.model.test_model(dataloader)

    def plot_performance(self) -> None:
        self.model.plot_performance()

    def save_model(self, filepath: str) -> None:
        """
        Saves model weights and optimizer state.

        Args:
            filepath (str): Path to save the checkpoint.
        """
        # Ensure models are on CPU before saving
        self.model.cpu()
        for client in self.clients:
            client.cpu()

        torch.save({
            'server_model_state_dict': self.model.state_dict(),
            'server_training_loss_history': self.model.training_loss_history,
            'server_training_accuracy_history': self.model.training_accuracy_history,
            'server_validate_loss_history': self.model.validate_loss_history,
            'server_validate_accuracy_history': self.model.validate_accuracy_history,
            'current_training_round': self.current_training_round,
            'client_states': [client.state_dict() for client in self.clients],
            'client_scores': [client.scores for client in self.clients] # Save client scores for TaLoS
        }, filepath)
        print(f"Server and clients saved to {filepath}")

        # Move server model back to original device
        self.model.to(device)
        # Clients will be moved to device when needed

    def load_model(self, filepath: str) -> None:
        """
        Loads model weights from a checkpoint.

        Args:
            filepath (str): Path to the saved checkpoint.
        """
        checkpoint = torch.load(filepath, map_location='cpu') # Load onto CPU first

        self.model.load_state_dict(checkpoint['server_model_state_dict'])
        self.model.training_loss_history = checkpoint['server_training_loss_history']
        self.model.training_accuracy_history = checkpoint['server_training_accuracy_history']
        self.model.validate_loss_history = checkpoint['server_validate_loss_history']
        self.model.validate_accuracy_history = checkpoint['server_validate_accuracy_history']
        self.current_training_round = checkpoint['current_training_round']

        client_states = checkpoint['client_states']
        client_scores = checkpoint.get('client_scores', [[] for _ in range(self.K)]) # Handle case where scores weren't saved

        for i in range(self.K):
            self.clients[i].load_state_dict(client_states[i])
            self.clients[i].scores = client_scores[i]


        # Move server model to target device
        self.model.to(device)
        # Clients will be moved to device when needed

        for client_id in range(self.K):
            self.clients[client_id].load_model(f"{filepath}{client_id}")
        print(f"Clients and server Model loaded from {filepath}")

    def recover_model(self, filepath: str) -> None:
        """
        Loads model weights and optimizer state from a checkpoint.

        Args:
            filepath (str): Path to the saved checkpoint.
        """
        self.model.recover_model(f"{filepath}server")
        self.current_training_round = self.model.current_epoch # jank, I know
        self.model.current_epoch = 1

        for client_id in range(self.K):
            self.clients[client_id].recover_model(f"{filepath}{client_id}")
        print(f"Clients and server Model loaded from {filepath}")


    def load_single_model(self,filepath: str) -> None:
        """
        Loads a single model checkpoint and applies it to all clients and the server.
        Useful for initializing all participants with the same pretrained model.
        """
        checkpoint = torch.load(filepath, map_location='cpu') # Load onto CPU first

        # Load into server model
        self.model.load_state_dict(checkpoint['model_state_dict'])
        # self.model.optimizer.load_state_dict(checkpoint['optimizer_state_dict']) # Load optimizer if needed

        # Load into each client model
        for client_id in range(self.K):
            self.clients[client_id].load_state_dict(checkpoint['model_state_dict'])
            # self.clients[client_id].optimizer.load_state_dict(checkpoint['optimizer_state_dict']) # Load optimizer if needed


        # Move server model to target device
        self.model.to(device)
        # Clients will be moved to device when needed

        print(f"Clients and server Model initialized from {filepath}")

In [10]:
'''
TODO:
Batch normalization, probably. I think I read somewhere that the facebook model needs this. someone also needs to look up how to make good use of the model
    - Added normalization transform to images from facebook github // Gustav
Version control and checkpointing.
    - added version control, checkpoitning left //Gustav
    - Added checkpointing. Not tested yet //Gustav
    - Tested and fully implemented // Gustav
Testing and hyperparameter tuning. also model head architecture
    - Added very simple model head architecture
gradient mask TaLoS thing. Also expanding this to FL. This is a big one.
    - possibly implemented for single model case. needs testing
Plot accuracy and loss method in DinoFullModel
    - added // Gustav
Type hinting
    - added // Gustav

Questions:
    - Is the sharding supposed to work like this?
        - is the iid supposed to have a uniform distribution of labels
        - explain non-iid
    - Is there a better way to do addition and multiplication in FedAvg?
    - Facebook model head. Is it correct? Should the backend be static?
    - are the results reasonable?
    - Batch normalization?
    - The aldready implemented normalization?

Personal contribution:
    - Quantization
    - Normalising parameters. they get very large. dont know why
    - could be done layer by layer so that a roughly even amount of parameters
        get pruned
    - sharing gradient mask
'''


torch.cuda.empty_cache()
def create_classifier():
    model = DinoFullModel(learning_rate=1e-3, momentum = 0.9).to(device)
    model.freeze_backbone()
    model.train_model(train_dataloader, 30, validate_dataloader=validate_dataloader, filepath="classifier")
    model.plot_performance()
    print("Done!")

def single_test():
    model = DinoFullModel(learning_rate=1e-3, momentum = 0.9).to(device)

    model.train_model(train_dataloader, 30, validate_dataloader=validate_dataloader, filepath="lr3m99e30cosine")
    model.plot_performance()
    print("Done!")

def finetune_test_single():
    fake_model = DinoFullModel(learning_rate=1e-3, momentum = 0.9, filepath="lr3m9e30cosine_sp1r3_1").to(device)
    model = DinoFullModel(learning_rate=1e-4, momentum = 0.9, filepath="classifier").to(device)
    #model.recover_model("lr3m9e30cosine_sp1r3_1")

    model.scores = fake_model.scores
    model.create_mask(0.9)
    model.drop_parameters()
    #model.apply_mask()
    #model.freeze_head()
    #model.unfreeze_backbone()
    path = "lr3m9e30cosine_sp1r3_1"
    R = 0.9
    model.TaLoS(train_dataloader, sparsity = R, rounds=3, q=50, filepath=path)

    model.apply_mask()
    model.train_model(train_dataloader, 30, validate_dataloader, filepath="test")
    model.plot_performance()

    print("Done!")

def FL_test_iid():
    FL = FL_server(K)
    #FL.load_single_model("baseline")
    FL.FedAvg(split_train_dataloader_iid, \
              FedAvg_rounds = 30, client_epochs = 4, C= 0.1, \
              validate_dataloader=validate_dataloader, \
              filepath="test_FLmodel")
    print(FL.test_model(test_dataloader))
    FL.plot_performance()

def FL_test_non_iid():
    FL = FL_server(K)
    #FL.load_single_model("baseline")
    FL.FedAvg(split_train_dataloader_non_iid, \
              FedAvg_rounds = 30, client_epochs = 4, C= 0.1, \
              validate_dataloader=validate_dataloader, \
              filepath="test_FLmodel")
    print(FL.test_model(test_dataloader))
    FL.plot_performance()

def FL_test_finetune():
    FL = FL_server(K).to(device)
    FL.load_single_model("baseline")
    FL.distributed_TaLos(split_train_dataloader_iid, \
                         sparisty=0.1, rounds = 3, \
                         filepath="test_FLmodel_finetune")
    FL.FedAvg(split_train_dataloader_iid, \
              FedAvg_rounds = 5, client_epochs = 5, C= 1, \
              validate_dataloader=validate_dataloader, \
              filepath="test_FLmodel_finetune")
    FL.plot_performance()


def FL_test_finetune_share():
    FL = FL_server(K).to(device)
    FL.load_single_model("baseline")
    FL.shared_mask_TaLoS(split_train_dataloader_iid, \
                         sparisty=0.1, rounds = 3, \
                         #q=5
                         filepath="test_FLmodel_shared_finetune")

    FL.FedAvg(split_train_dataloader_iid, \
              FedAvg_rounds = 5, client_epochs = 5, C= 1, \
              validate_dataloader=validate_dataloader, \
              filepath="test_FLmodel_shared_finetune")
    FL.plot_performance()

def test_quantizer():
    model = DinoFullModel().to(device)
    arr = torch.randn(3,3)
    model.scores = [arr]
    print(model.scores)
    print(model.get_quantized_scores(2))

#test_quantizer()
#create_classifier()
#finetune_test_single()
FL_test_iid()
FL_test_non_iid()

#model = DinoFullModel(learning_rate=1e-3, momentum = 0.9,epochs = 30, filepath="lr3m9e30cosine_sp1r3_2").to(device)
#model.test_mask_quantization(0.4641588833612779, 100)

Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main
Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main
Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main
Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main
Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main
Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main
Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main
Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main
Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main
Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main
Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main
Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main
Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main
Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main
Using 

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 16.12 MiB is free. Process 13230 has 14.72 GiB memory in use. Of the allocated memory 13.76 GiB is allocated by PyTorch, and 847.98 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)